In [ ]:
# CNN Training

In this notebook we train a Convolutional Neural Network (CNN) to classify
financial chart images into two classes:

- up
- down

The model is intentionally simple and serves as a baseline to study
the feasibility and limitations of image-based technical pattern detection.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam

from sklearn.utils.class_weight import compute_class_weight


In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 20

BASE_DIR = "data"
TRAIN_DIR = os.path.join(BASE_DIR, "train", "images")
VAL_DIR   = os.path.join(BASE_DIR, "val", "images")


In [ ]:
train_gen = ImageDataGenerator(
    rescale=1./255,
    horizontal_flip=False
)

val_gen = ImageDataGenerator(rescale=1./255)

train_data = train_gen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=True
)

val_data = val_gen.flow_from_directory(
    VAL_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False
)

train_data.class_indices


In [ ]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_data.classes),
    y=train_data.classes
)

class_weight = dict(enumerate(class_weights))
class_weight


In [ ]:
model = Sequential([
    Conv2D(32, 3, activation="relu", input_shape=(224, 224, 3)),
    MaxPooling2D(),

    Conv2D(64, 3, activation="relu"),
    MaxPooling2D(),

    Conv2D(128, 3, activation="relu"),
    MaxPooling2D(),

    Flatten(),
    Dense(128, activation="relu"),
    Dropout(0.5),
    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()


In [ ]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=EPOCHS,
    class_weight=class_weight,
    verbose=1
)


In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Train")
plt.plot(history.history["val_accuracy"], label="Validation")
plt.title("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Train")
plt.plot(history.history["val_loss"], label="Validation")
plt.title("Loss")
plt.legend()

plt.show()


In [ ]:
os.makedirs("models", exist_ok=True)
model.save("models/cnn_chart_model.keras")


In [ ]:
## Training Summary

- The model converges quickly but shows unstable validation behavior.
- Accuracy alone is misleading due to class imbalance.
- The network tends to collapse toward a single dominant class.
- This confirms the difficulty of detecting technical patterns from raw chart images.

Further evaluation is required using detailed classification metrics.
